In [22]:
import sys
import os
import urllib3

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)

Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [23]:
import pandas as pd
from datetime import datetime, timedelta
import pytz

# Define time period
# Calculate the start date (2 days ago) at 00:00:00 UTC
start_date = (datetime.now(pytz.UTC) - timedelta(days=2)).date()

# Format as 'YYYY-MM-DDT00:00:00Z'
start = f"{start_date}T00:00:00Z"

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        observed_src = observed_src[observed_src['lastObserved'] >= start]
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")

display(observed_src)


Querying owner: HTOC Org

Retrieved 751 unique observed indicators.

Retrieved 751 unique observed indicators.


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,sha256,source,sha1,Block,text,md5,size,address,url,indicator
6,6755399443033810,2025-04-11T17:46:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-08T12:13:37Z,4.0,67.0,(U//FOUO) Federal law enforcement and the Inte...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,190.92.174.130
36,5629499555060731,2025-06-16T17:42:54Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-08T11:54:50Z,3.0,64.0,RITM0589984,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.220.101.17
51,6755399460008312,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-08T11:06:23Z,4.0,60.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.135.79.91
52,6755399460008397,2025-07-02T12:05:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-08T11:02:58Z,4.0,60.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,190.242.157.230
53,6755399460007924,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-08T11:02:30Z,4.0,60.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.111.244.210
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9121,6755399460007437,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:12Z,4.0,60.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.28.200.170
9153,6755399460007783,2025-07-02T12:05:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:10Z,4.0,60.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,181.174.115.9
9179,6755399460007842,2025-07-02T12:05:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:09Z,4.0,60.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.237.210.82
9217,6755399460008013,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-05T01:35:07Z,4.0,60.0,"Details\nIn late June 2025, Iran-aligned hackt...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122.54.133.163


In [24]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251008.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251007.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251006.csv']
Loaded data from 3 files.
Loaded data from 3 files.


C:\Users\jaskew\AppData\Local\Temp\ipykernel_356\564055639.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


In [25]:
import pandas as pd
from datetime import timedelta
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION & SETUP
# ═══════════════════════════════════════════════════════════════════════════════

# Time cutoffs
cutoff = pd.Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=180)
cutoff_naive = cutoff.tz_convert(None)

# Required columns validation
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [c for c in required_columns if c not in observed_data_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# ═══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def process_tag_row(row, observed_src):
    """Process a single row from observed_src to extract and filter tags."""
    tags_data = row.get('tags.data')
    if not isinstance(tags_data, list):
        return None

    # Normalize the tags for the current row
    tags_df = pd.json_normalize(tags_data)

    # Ensure string and apply VA rename before filtering
    tags_df['name'] = (
        tags_df['name']
        .astype(str)
        .str.replace('VA CSOC CTS Splunk', 'VA Splunk API', regex=False)
    )

    # Skip if all associated groups are Adversary
    if 'associatedGroups.data' in observed_src.columns:
        ag_data = row.get('associatedGroups.data')
        if isinstance(ag_data, list) and len(ag_data) > 0:
            groups_df = pd.json_normalize(ag_data)
            if 'type' in groups_df.columns and set(groups_df['type']) == {'Adversary'}:
                return None

    # All tag names (for all_tags)
    all_tags_list = tags_df['name'].tolist()

    # Filter for API tags
    api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)].copy()
    if api_tags.empty:
        return None

    # Add metadata columns
    meta_cols = [
        'summary', 'observations', 'description', 'type',
        'dateAdded', 'lastModified', 'lastObserved', 'webLink',
        'rating', 'confidence', 'id', 'associatedGroups.data'
    ]
    for col in meta_cols:
        api_tags[col] = [row.get(col)] * len(api_tags)

    # Attach all tags list
    if len(api_tags) > 0:
        api_tags['all_tags'] = [all_tags_list] * len(api_tags)

    return api_tags

def map_observed_dates(filtered_tags, observed_data_df):
    """Map observed dates from observed_data_df to filtered_tags."""
    if filtered_tags.empty:
        return filtered_tags
    
    # Clean name -> match OpDiv (remove trailing ' Splunk API')
    filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)
    filtered_tags['observed_date'] = pd.NaT
    
    # Ensure observed_data_df obs_date is datetime (naive)
    observed_data_df['obs_date'] = pd.to_datetime(observed_data_df['obs_date'], errors='coerce')

    for idx, r in filtered_tags.iterrows():
        summary = r.get('summary')
        cleaned_name = r.get('cleaned_name')
        if pd.isna(summary) or pd.isna(cleaned_name):
            continue
        match = observed_data_df[
            (observed_data_df['indicator'] == summary) &
            (observed_data_df['OpDiv'] == cleaned_name)
        ]
        if not match.empty:
            filtered_tags.at[idx, 'observed_date'] = match['obs_date'].iloc[0]

    filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')
    
    # Drop helper column
    filtered_tags.drop(columns=['cleaned_name'], inplace=True, errors='ignore')
    
    return filtered_tags

def apply_filters_and_grouping(filtered_tags, cutoff, date_added_cutoff, cutoff_naive):
    """Apply time filters, partner grouping, and other filtering criteria."""
    if filtered_tags.empty:
        return pd.DataFrame()
    
    # Time-based filters
    # Last 24h by lastObserved (aware) - but we'll use the 2-day filter below
    # recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - timedelta(hours=24)].copy()
    
    # Last 2 days by observed_date (naive)
    recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=2)].copy()

    if recent_tags.empty:
        return recent_tags
    
    # Partner extraction and grouping
    recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

    partner_groups = (
        recent_tags.groupby('summary')['partner']
        .agg(['nunique', lambda s: ', '.join(sorted(set(s.dropna())))]).reset_index()
        .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
    )

    recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

    # Additional filters
    recent_tags = recent_tags[recent_tags['partner_count'] >= 2]
    recent_tags = recent_tags[recent_tags['observations'] < 15000]
    recent_tags = recent_tags[recent_tags['dateAdded'] >= date_added_cutoff]

    # Deduplicate by summary
    recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

    # Drop unneeded columns if present
    cols_to_drop = [
        'techniqueId', 'tactics.data', 'tactics.count',
        'platforms.data', 'platforms.count', 'partner', 'name'
    ]
    recent_tags = recent_tags.drop(columns=[c for c in cols_to_drop if c in recent_tags.columns], errors='ignore')

    # Remove rows where all_tags contains unwanted markers
    if 'all_tags' in recent_tags.columns:
        recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'I&W' in x)]
        recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'htoc_wl' in x)]

    return recent_tags

def extract_group_ids(recent_tags):
    """Extract group IDs from associatedGroups.data."""
    if 'associatedGroups.data' in recent_tags.columns:
        recent_tags['group_id'] = recent_tags['associatedGroups.data'].apply(
            lambda x: x[0]['id'] if isinstance(x, list) and len(x) > 0 and 'id' in x[0] else None
        )
        # Convert group_id to string type and remove trailing decimals if it exists
        if 'group_id' in recent_tags.columns:
            recent_tags['group_id'] = recent_tags['group_id'].apply(
                lambda x: str(int(float(x))) if pd.notna(x) and str(x) != 'None' else x
            ).astype(str)
    return recent_tags

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN PROCESSING PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

print("Starting tag processing pipeline...")

# Step 1: Process tags from observed_src
print("Processing tags from observed_src...")
all_filtered = []

for _, row in observed_src.iterrows():
    processed_tags = process_tag_row(row, observed_src)
    if processed_tags is not None:
        all_filtered.append(processed_tags)

# Step 2: Create filtered_tags DataFrame
print("Creating filtered_tags DataFrame...")
filtered_tags = pd.concat(all_filtered, ignore_index=True) if all_filtered else pd.DataFrame()

if not filtered_tags.empty:
    # Ensure datetime columns
    filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce', utc=True)
    filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce', utc=True)
    print(f"Created filtered_tags with {len(filtered_tags)} rows")
else:
    print("No filtered tags found")

# Step 3: Map observed dates
print("Mapping observed dates...")
filtered_tags = map_observed_dates(filtered_tags, observed_data_df)

# Step 4: Apply filters and grouping
print("Applying filters and partner grouping...")
recent_tags = apply_filters_and_grouping(filtered_tags, cutoff, date_added_cutoff, cutoff_naive)

# Step 5: Extract group IDs
print("Extracting group IDs...")
recent_tags = extract_group_ids(recent_tags)

# Final summary
print(f"Processing complete! Final dataset shape: {recent_tags.shape}")
if not recent_tags.empty:
    print(f"Partners represented: {recent_tags['partners'].nunique()} unique partner combinations")
    print(f"Date range: {recent_tags['observed_date'].min()} to {recent_tags['observed_date'].max()}")

recent_tags.head(20)

Starting tag processing pipeline...
Processing tags from observed_src...
Creating filtered_tags DataFrame...
Creating filtered_tags DataFrame...
Created filtered_tags with 5782 rows
Mapping observed dates...
Created filtered_tags with 5782 rows
Mapping observed dates...
Applying filters and partner grouping...
Extracting group IDs...
Processing complete! Final dataset shape: (33, 18)
Partners represented: 20 unique partner combinations
Date range: 2025-10-07 00:00:00 to 2025-10-08 00:00:00
Applying filters and partner grouping...
Extracting group IDs...
Processing complete! Final dataset shape: (33, 18)
Partners represented: 20 unique partner combinations
Date range: 2025-10-07 00:00:00 to 2025-10-08 00:00:00


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id
2,5629499555060731,2025-10-07T07:34:30Z,RITM0589984,185.220.101.17,9169,Address,2025-06-16 17:42:54+00:00,2025-10-08T11:54:50Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,64.0,NaN,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-10-08,2,"CMS, HHS",nan
10,6755399461914247,2025-10-08T00:09:42Z,INC9137833,scan.cypex.ai,7446,Host,2025-07-09 14:26:10+00:00,2025-10-08T07:40:02Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,67.0,NaN,"[VA Splunk API, NIH Splunk API, IHS Splunk API...",2025-10-08,2,"IHS, VA",nan
13,6755399468016137,2025-10-08T00:09:42Z,"Domain is nonmalicious, should be exempt from ...",pmstudycircle.com,1129,Host,2025-08-20 16:02:13+00:00,2025-10-08T07:39:57Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,73.0,NaN,"[DHA Splunk API, CMS Splunk API, VA Splunk API...",2025-10-08,2,"HRSA, VA",nan
348,6755399447111239,2025-10-07T09:09:13Z,FBI Email Alert May 14 2025,104.152.52.110,1875,Address,2025-05-14 17:44:46+00:00,2025-10-08T07:39:43Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,60.0,NaN,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-10-07,3,"HHS, OS, VA",nan
531,5629499541090468,2025-10-07T09:09:13Z,FBI Email Alert May 14 2025 Medium IP IOCs,104.152.52.139,1395,Address,2025-05-14 17:47:33+00:00,2025-10-08T07:39:40Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,61.0,NaN,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-10-07,3,"HHS, OS, VA",nan
673,5629499537014343,2025-10-08T00:09:42Z,CISA observed scripted download and execution ...,104.21.20.66,573,Address,2025-04-21 11:07:48+00:00,2025-10-08T07:39:38Z,2025-10-07 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,51.0,NaN,"[DHA Splunk API, malicious code, VA OIS CSOC C...",2025-10-07,3,"IHS, NIH, VA",nan
690,5629499542008908,2025-10-08T11:02:30Z,RITM0581773/TASK0877780,89.149.52.55,1717,Address,2025-05-19 11:46:06+00:00,2025-10-08T07:39:37Z,2025-10-07 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,61.0,NaN,"[DHA Splunk API, CMS Splunk API, VA Splunk API...",2025-10-07,2,"CDC, IHS",nan
732,6755399459033767,2025-10-07T07:34:30Z,RITM0589984,45.141.215.62,4799,Address,2025-06-16 17:42:47+00:00,2025-10-08T07:39:36Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,64.0,NaN,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-10-08,2,"CMS, HHS",nan
831,6755399459033734,2025-10-07T19:11:04Z,RITM0589984,195.47.238.83,5034,Address,2025-06-16 17:42:22+00:00,2025-10-08T07:39:34Z,2025-10-08 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,64.0,NaN,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-10-07,2,"CMS, FDA",nan
1158,6755399470368159,2025-10-08T11:06:23Z,IRT 67469,washingtonopenmri.com,551,Host,2025-09-02 14:50:08+00:00,2025-10-08T05:01:26Z,2025-10-07 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,75.0,NaN,"[DHA Splunk API, CMS Splunk API, VA Splunk API...",2025-10-07,2,"DHA, VA",nan


In [26]:
import pandas as pd

# Extract unique indicators from recent_tags
indicators = recent_tags['summary'].unique()

# Initialize a list to store attribute data
attributes_data = []

# Track indicators with no attributes
indicators_with_no_attributes = []

# Iterate over unique indicators
for indicator in indicators:
    try:
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{indicator}?fields=attributes&resultStart=0&resultLimit=1000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            data = response.json().get('data', {})
            attributes = data.get('attributes', {}).get('data', [])

            if not attributes:
                print(f"No attributes found for indicator: {indicator}")
                # Track indicators with no attributes
                indicators_with_no_attributes.append(indicator)
            else:
                for attr in attributes:
                    attr.update({
                        'id': data.get('id'),
                        'summary': data.get('summary'),
                        'type': data.get('type'),
                        'ownerName': data.get('ownerName')
                    })
                    attributes_data.append(attr)
        # Suppress non-JSON and known 400 error responses
    except Exception as e:
        # Suppress error output, including known 400 error
        if hasattr(e, 'response') and getattr(e.response, 'status_code', None) == 400:
            continue
        if "Status Code: 400" in str(e):
            continue
        pass

# Create attributes 
attributes_observed_src = pd.json_normalize(attributes_data)

# Un-nest 'createdBy' and filter out 'SOAR' entries
if not attributes_observed_src.empty and attributes_observed_src['createdBy.lastName'].notnull().any():
    attributes_observed_src = attributes_observed_src[attributes_observed_src['createdBy.lastName'] != 'SOAR']

# Drop duplicates based on 'id' to keep unique attribute records
attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

# Filter recent_tags for indicators that had attributes
filtered_with_attrs = recent_tags[recent_tags['summary'].isin(attributes_observed_src['summary'])]

# Filter recent_tags for indicators that had no attributes
no_attrs_df = recent_tags[recent_tags['summary'].isin(indicators_with_no_attributes)]

# Combine both and remove duplicates based on 'summary'
filtered_recent_tags = pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
filtered_recent_tags = filtered_recent_tags.drop_duplicates(subset='summary').reset_index(drop=True)
display(filtered_recent_tags)


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id
0,5629499537014343,2025-10-08T00:09:42Z,CISA observed scripted download and execution ...,104.21.20.66,573,Address,2025-04-21 11:07:48+00:00,2025-10-08T07:39:38Z,2025-10-07 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,51.0,NaN,"[DHA Splunk API, malicious code, VA OIS CSOC C...",2025-10-07,3,"IHS, NIH, VA",nan
1,6755399460007636,2025-10-08T00:09:42Z,"Details\nIn late June 2025, Iran-aligned hackt...",47.251.43.115,787,Address,2025-07-02 12:05:30+00:00,2025-10-08T02:34:11Z,2025-10-07 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,2.0,60.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-10-07,2,"HHS, VA",5629499544001057
2,6755399460007435,2025-10-07T17:41:33Z,"Details\nIn late June 2025, Iran-aligned hackt...",104.28.155.218,291,Address,2025-07-02 12:05:29+00:00,2025-10-07T15:44:34Z,2025-10-07 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,60.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-10-07,2,"HRSA, NIH",5629499544001057
3,6755399460008045,2025-10-08T11:06:23Z,"Details\nIn late June 2025, Iran-aligned hackt...",157.15.62.44,269,Address,2025-07-02 12:05:34+00:00,2025-10-06T01:55:26Z,2025-10-07 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,60.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-10-07,2,"DHA, VA",5629499544001057


In [28]:
# Save all columns from filtered_recent_tags to Excel with clickable hyperlinks for 'webLink'
output_dir = r"Z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Spreadsheet"
os.makedirs(output_dir, exist_ok=True)
excel_path = os.path.join(
    output_dir,
    f"I&W_indicators_full_{pd.Timestamp.now().strftime('%Y%m%d')}.xlsx"
)

try:
    import openpyxl
    from openpyxl.styles import Font

    # Prepare data for Excel - convert all complex data types to strings
    excel_data = filtered_recent_tags.copy()
    
    # Convert timezone-aware datetime columns to timezone-naive
    for col in excel_data.columns:
        if excel_data[col].dtype == 'datetime64[ns, UTC]' or (excel_data[col].dtype == 'object' and 
                                                             excel_data[col].apply(lambda x: hasattr(x, 'tzinfo') and x.tzinfo is not None).any()):
            excel_data[col] = pd.to_datetime(excel_data[col], errors='coerce').dt.tz_convert(None)
    
    # Convert complex data types to strings
    for col in excel_data.columns:
        if excel_data[col].dtype == 'object':  # Check if column might contain complex objects
            excel_data[col] = excel_data[col].apply(
                lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x) if x is not None else ''
            )

    # Create a new workbook and worksheet
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "I&W Indicators Full"

    # Write headers
    for col_idx, col_name in enumerate(excel_data.columns, 1):
        ws.cell(row=1, column=col_idx, value=col_name)

    # Write data rows
    for row_idx, row in enumerate(excel_data.itertuples(index=False), 2):
        for col_idx, value in enumerate(row, 1):
            col_name = excel_data.columns[col_idx - 1]
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            # Make 'webLink' column clickable
            if col_name == 'webLink' and pd.notna(value) and value != '':
                cell.hyperlink = value
                cell.font = Font(color="0563C1", underline="single")

    # Auto-adjust column widths
    for column_cells in ws.columns:
        max_length = 0
        column_letter = column_cells[0].column_letter
        for cell in column_cells:
            try:
                if cell.value is not None and len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except Exception:
                pass
        ws.column_dimensions[column_letter].width = min(max_length + 2, 50)

    wb.save(excel_path)
    print(f"Saved all filtered_recent_tags data to Excel: {excel_path}")

except ImportError:
    print("openpyxl not available. Install with: pip install openpyxl")
    print("Excel file with hyperlinks not created.")
except Exception as e:
    print(f"Error creating Excel file: {e}")

Saved all filtered_recent_tags data to Excel: Z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Spreadsheet\I&W_indicators_full_20251008.xlsx
